# Companies House cleaning, multi-SIC and 8-sector Match 

- standardizes column names and values
- format dates and numeric fields
- extracts up to 4 SIC codes per company
- matches every SIC code to a sector
- produces company-level summaries with a primary industry and all matched sectors
- filter CountryOfOrigin + add flag
- exports to CSV

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path


In [2]:
# Input / output paths
INPUT = Path("../data/BasicCompanyDataAsOneFile-2026-06-01.csv")   

# Read as strings first to preserve IDs and avoid accidental type coercion
df_raw = pd.read_csv(INPUT, dtype=str)
df_raw.head(3)


,CompanyName,CompanyNumber,RegAddress.CareOf,RegAddress.POBox,RegAddress.AddressLine1,RegAddress.AddressLine2,RegAddress.PostTown,RegAddress.County,RegAddress.Country,RegAddress.PostCode,...,PreviousName_7.CONDATE,PreviousName_7.CompanyName,PreviousName_8.CONDATE,PreviousName_8.CompanyName,PreviousName_9.CONDATE,PreviousName_9.CompanyName,PreviousName_10.CONDATE,PreviousName_10.CompanyName,ConfStmtNextDueDate,ConfStmtLastMadeUpDate
0,! LTD,08209948,NaN,NaN,9 PRINCES SQUARE,NaN,HARROGATE,NaN,ENGLAND,HG1 1ND,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25/09/2026,11/09/2025
1,!ABRIDGE TAX LTD,16092999,NaN,NaN,82 GREAT NORTH ROAD,GREAT NORTH BUSINESS CENTRE,HATFIELD,NaN,UNITED KINGDOM,AL9 5BL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,04/12/2026,20/11/2025
2,!BIG IMPACT GRAPHICS LIMITED,11743365,NaN,NaN,124 CITY ROAD,NaN,LONDON,NaN,NaN,EC1V 2NX,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29/12/2026,15/12/2025


In [3]:
len(df_raw)

5698274

In [4]:

def clean_column_name(name: str) -> str:
    return str(name).strip().replace(".", "_").replace(" ", "_")

def strip_if_str(x):
    if isinstance(x, str):
        x = x.strip()
        if x in {"", "nan", "None", "NULL", "null"}:
            return pd.NA
        return x
    return x

df = df_raw.copy()
df.columns = [clean_column_name(c) for c in df.columns]
df = df.apply(lambda col: col.map(strip_if_str))

df.head(3)


,CompanyName,CompanyNumber,RegAddress_CareOf,RegAddress_POBox,RegAddress_AddressLine1,RegAddress_AddressLine2,RegAddress_PostTown,RegAddress_County,RegAddress_Country,RegAddress_PostCode,...,PreviousName_7_CONDATE,PreviousName_7_CompanyName,PreviousName_8_CONDATE,PreviousName_8_CompanyName,PreviousName_9_CONDATE,PreviousName_9_CompanyName,PreviousName_10_CONDATE,PreviousName_10_CompanyName,ConfStmtNextDueDate,ConfStmtLastMadeUpDate
0,! LTD,08209948,NaN,NaN,9 PRINCES SQUARE,NaN,HARROGATE,NaN,ENGLAND,HG1 1ND,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25/09/2026,11/09/2025
1,!ABRIDGE TAX LTD,16092999,NaN,NaN,82 GREAT NORTH ROAD,GREAT NORTH BUSINESS CENTRE,HATFIELD,NaN,UNITED KINGDOM,AL9 5BL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,04/12/2026,20/11/2025
2,!BIG IMPACT GRAPHICS LIMITED,11743365,NaN,NaN,124 CITY ROAD,NaN,LONDON,NaN,NaN,EC1V 2NX,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29/12/2026,15/12/2025


In [5]:
len(df)

5698274

In [6]:
df.columns

Index(['CompanyName', 'CompanyNumber', 'RegAddress_CareOf', 'RegAddress_POBox',
       'RegAddress_AddressLine1', 'RegAddress_AddressLine2',
       'RegAddress_PostTown', 'RegAddress_County', 'RegAddress_Country',
       'RegAddress_PostCode', 'CompanyCategory', 'CompanyStatus',
       'CountryOfOrigin', 'DissolutionDate', 'IncorporationDate',
       'Accounts_AccountRefDay', 'Accounts_AccountRefMonth',
       'Accounts_NextDueDate', 'Accounts_LastMadeUpDate',
       'Accounts_AccountCategory', 'Returns_NextDueDate',
       'Returns_LastMadeUpDate', 'Mortgages_NumMortCharges',
       'Mortgages_NumMortOutstanding', 'Mortgages_NumMortPartSatisfied',
       'Mortgages_NumMortSatisfied', 'SICCode_SicText_1', 'SICCode_SicText_2',
       'SICCode_SicText_3', 'SICCode_SicText_4',
       'LimitedPartnerships_NumGenPartners',
       'LimitedPartnerships_NumLimPartners', 'URI', 'PreviousName_1_CONDATE',
       'PreviousName_1_CompanyName', 'PreviousName_2_CONDATE',
       'PreviousName_2_Compan

In [7]:

def parse_date(series):
    return pd.to_datetime(series, dayfirst=True, errors="coerce")

def extract_sic_code(value):
    """
    Extract a 5-digit SIC code from a raw SIC text field.
    Returns pd.NA for missing / invalid values.
    """
    if pd.isna(value):
        return pd.NA
    s = str(value).strip()
    if s.lower() in {"none supplied", "n/a", "na", "none"}:
        return pd.NA
    m = re.search(r"(\d{5})", s)
    if not m:
        return pd.NA
    return m.group(1)

def extract_sic_desc(value):
    """
    Return the description part after 'code - ' if present.
    """
    if pd.isna(value):
        return pd.NA
    s = str(value).strip()
    if s.lower() in {"none supplied", "n/a", "na", "none"}:
        return pd.NA
    if " - " in s:
        return s.split(" - ", 1)[1].strip()
    return s



In [8]:
# #get Sic codes
# sic_cols = [
#     "SICCode_SicText_1",
#     "SICCode_SicText_2",
#     "SICCode_SicText_3",
#     "SICCode_SicText_4"
# ]

# all_sic_values = (
#     pd.concat([df[col] for col in sic_cols])
#       .dropna()
#       .astype(str)
#       .str.strip()
#       .unique()
# )


# unique_sic_df = pd.DataFrame({
#     "sic_raw": sorted(all_sic_values)
# })

# unique_sic_df.to_csv(
#     "unique_sic_values.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

In [9]:

FAST_GROWTH_CODES = {
    "58290",  # Software publishing
    "62012",  # Business and domestic software development
    "62020",  # Information technology consultancy activities
    "62090",  # Other information technology service activities
    "63110",  # Data processing, hosting and related activities
    "63120",  # Web portals
    "72110",  # Research and experimental development on biotechnology
    "72190",  # Other research and experimental development
}


def code_to_sector(sic_code):
    """
    Input:
        sic_code: 5-digit SIC code

    Returns:
        (sector, is_fast_growth)
    """

    if pd.isna(sic_code):
        return ("Other", False)

    sic = str(sic_code).strip().zfill(5)

    # ----------------------------------
    # Fast Growth & Emerging Sector
    # ----------------------------------
    if sic in FAST_GROWTH_CODES:
        return (
            "Fast growth & emerging sector",
            True
        )

    try:
        div = int(sic[:2])
    except:
        return ("Other", False)

    # ----------------------------------
    # Agriculture
    # SIC 01-03
    # ----------------------------------
    if 1 <= div <= 3:
        return (
            "Agriculture",
            False
        )

    # ----------------------------------
    # Manufacturing
    # SIC 10-33
    # ----------------------------------
    if 10 <= div <= 33:
        return (
            "Manufacturing",
            False
        )

    # ----------------------------------
    # Wholesale & Retail
    # SIC 45-47
    # ----------------------------------
    if 45 <= div <= 47:
        return (
            "Wholesale & Retail",
            False
        )

    # ----------------------------------
    # Real Estate
    # SIC 68
    # ----------------------------------
    if div == 68:
        return (
            "Real Estate",
            False
        )

    # Residents Property Management
    if sic == "98000":
        return (
            "Real Estate",
            False
        )

    # ----------------------------------
    # Technology, Legal & Professional
    # SIC 58-63
    # SIC 69-75
    # ----------------------------------
    if (58 <= div <= 63) or (69 <= div <= 75):
        return (
            "Technology, legal & professional",
            False
        )

    # ----------------------------------
    # Healthcare
    # SIC 86-88
    # ----------------------------------
    if 86 <= div <= 88:
        return (
            "Healthcare",
            False
        )

    # ----------------------------------
    # Public sector, education & charities
    # SIC 84
    # SIC 85
    # SIC 94
    # SIC 91
    # ----------------------------------
    if div in [84, 85, 94, 91]:
        return (
            "Public sector, education & charities",
            False
        )

    # ----------------------------------
    # Household Activities
    # SIC 98100 / 98200
    # ----------------------------------
    if div == 98:
        return (
            "Other",
            False
        )

    # ----------------------------------
    # Extraterritorial Organisations
    # SIC 99000
    # Dormant Company 99999
    # ----------------------------------
    if div == 99:
        return (
            "Other",
            False
        )

    # ----------------------------------
    # Everything else
    # ----------------------------------
    return (
        "Other",
        False
    )

In [10]:
# Clean and normalize common source fields
if "CompanyNumber" in df.columns:
    df["CompanyNumber"] = (
        df["CompanyNumber"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

# Parse date fields
date_cols = [
    "DissolutionDate",
    "IncorporationDate",
    "Accounts_NextDueDate",
    "Accounts_LastMadeUpDate",
    "Returns_NextDueDate",
    "Returns_LastMadeUpDate",
    "ConfStmtNextDueDate",
    "ConfStmtLastMadeUpDate",
]
date_cols += [f"PreviousName_{i}_CONDATE" for i in range(1, 11)]

for c in date_cols:
    if c in df.columns:
        df[c] = parse_date(df[c])

# Process numeric fields
num_cols = [
    "Accounts_AccountRefDay",
    "Accounts_AccountRefMonth",
    "Mortgages_NumMortCharges",
    "Mortgages_NumMortOutstanding",
    "Mortgages_NumMortPartSatisfied",
    "Mortgages_NumMortSatisfied",
    "LimitedPartnerships_NumGenPartners",
    "LimitedPartnerships_NumLimPartners",
]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

# make helper fields

today = pd.Timestamp.today().normalize()

inc = pd.to_datetime(df["IncorporationDate"], errors="coerce")

inc = inc.where((inc >= pd.Timestamp("1800-01-01")) & (inc <= today))
df["company_age_years"] = ((today - inc).dt.days / 365.25).round(1)

df["has_outstanding_charges"] = (
    df["Mortgages_NumMortOutstanding"] > 0
)

if "CompanyStatus" in df.columns:
    df["is_active"] = df["CompanyStatus"].eq("Active")

if "CompanyCategory" in df.columns:
    df["company_type_group"] = np.select(
        [
            df["CompanyCategory"].eq("Private Limited Company"),
            df["CompanyCategory"].eq("Limited Liability Partnership"),
            df["CompanyCategory"].eq("Community Interest Company"),
        ],
        ["Ltd", "LLP", "CIC"],
        default="Other"
    )

In [11]:
# Extract SICCodes from SicText_1~4 and build company SIC lists

sic_cols = [
    "SICCode_SicText_1",
    "SICCode_SicText_2",
    "SICCode_SicText_3",
    "SICCode_SicText_4",
]

def extract_sic_code(value):
    """
    Extract 5-digit SIC numeric code from raw SIC text(string).
    """
    if pd.isna(value):
        return pd.NA

    s = str(value).strip()
    if s == "" or s.lower() in {"none supplied", "na", "n/a", "null", "none"}:
        return pd.NA

    m = re.search(r"(\d{5})", s)
    if m:
        return m.group(1)

    return pd.NA

def row_unique_list(values):
    """Return ordered unique non-null values."""
    out = []
    seen = set()
    for v in values:
        if pd.isna(v):
            continue
        if v not in seen:
            seen.add(v)
            out.append(v)
    return out

# Extract SIC codes into helper columns
for c in sic_cols:
    if c in df.columns:
        df[c + "_code"] = df[c].map(extract_sic_code)

sic_code_cols = [c + "_code" for c in sic_cols if c in df.columns]

# Company-level SIC list
df["sic_codes_all"] = df[sic_code_cols].apply(
    lambda row: row_unique_list(row.tolist()),
    axis=1
)

# First valid SIC as the primary SIC
df["primary_sic_code"] = df["sic_codes_all"].apply(
    lambda x: x[0] if len(x) > 0 else pd.NA
)

# Keep only valid SICs for matching if you want to exclude dormant code
df["sic_codes_valid"] = df["sic_codes_all"].apply(
    lambda xs: [x for x in xs if x not in {"99999"}]
)

# Multi-SIC flag
df["multi_sic_company"] = df["sic_codes_valid"].apply(lambda x: len(x) > 1)

# Match each SIC to a sector
df["matched_sectors"] = df["sic_codes_valid"].apply(
    lambda codes: row_unique_list([code_to_sector(code)[0] for code in codes if code_to_sector(code)[0] != "Other"])
)

# Fast-growth flag if any SIC hits fast-growth codes
df["has_fast_growth"] = df["sic_codes_valid"].apply(
    lambda codes: any(code_to_sector(code)[1] for code in codes)
)

# Primary sector: fast-growth override first, then first matched sector, then Other
def choose_primary_sector(row):
    if row["has_fast_growth"]:
        return "Fast growth & emerging sector"
    if len(row["matched_sectors"]) > 0:
        return row["matched_sectors"][0]
    return "Other"

df["primary_sector"] = df.apply(choose_primary_sector, axis=1)

In [12]:
len(df)

5698274

In [13]:
df["CountryOfOrigin"].value_counts(dropna=False)

CountryOfOrigin
United Kingdom             5653357
VIRGIN ISLANDS, BRITISH       8577
JERSEY                        7133
ISLE OF MAN                   3602
UNITED STATES                 3355
                            ...   
MADAGASCAR                       1
UK                               1
LESOTHO                          1
Turkey                           1
YEMEN ARAB REPUBLIC              1
Name: count, Length: 228, dtype: int64

In [14]:
# Filter the 8 sectors

TARGET_SECTORS = [
    "Manufacturing",
    "Public sector, education & charities",
    "Healthcare",
    "Technology, legal & professional",
    "Agriculture",
    "Real Estate",
    "Wholesale & Retail",
    "Fast growth & emerging sector",
]

SECTOR_MAP = {
    "Agriculture": 1,
    "Manufacturing": 2,
    "Wholesale & Retail": 3,
    "Real Estate": 4,
    "Technology, legal & professional": 5,
    "Healthcare": 6,
    "Public sector, education & charities": 7,
    "Fast growth & emerging sector": 8,
}

# Sectors Numeric label
df["sector_id"] = df["primary_sector"].map(SECTOR_MAP)


# Keep only companies in the 8 target sectors
df_target = df[df["primary_sector"].isin(TARGET_SECTORS)].copy()

# deduplicate by CompanyNumber
if "CompanyNumber" in df_target.columns:
    df_target = df_target.sort_values(
        by=["CompanyNumber", "primary_sector"],
        kind="stable"
    ).drop_duplicates(subset=["CompanyNumber"], keep="first")

# keep a clean analysis table
keep_cols = [
    "CompanyNumber",
    "CompanyName",
    "CompanyStatus",
    "CompanyCategory",
    "CountryOfOrigin",       
    "RegAddress_Country", 
    "RegAddress_PostTown",
    "RegAddress_PostCode",   
    "IncorporationDate",
    "primary_sic_code",
    "sic_codes_all",
    "matched_sectors",
    "has_fast_growth",
    "multi_sic_company",
    "primary_sector",
    "sector_id",

    "Accounts_AccountCategory",
    "Accounts_LastMadeUpDate",
    "Accounts_NextDueDate",
    "Mortgages_NumMortOutstanding",
    "Mortgages_NumMortCharges",
    "Mortgages_NumMortSatisfied",
    
    "has_outstanding_charges",
    "company_age_years",

]

keep_cols = [c for c in keep_cols if c in df_target.columns]

df_target_clean = df_target[keep_cols].copy()


In [15]:
len(df_target)

3415689

In [16]:

df_target["CountryOfOrigin"].value_counts(dropna=False)

CountryOfOrigin
United Kingdom    3415686
UNITED KINGDOM          2
LUXEMBOURG              1
Name: count, dtype: int64

In [17]:
len(df_target_clean)

3415689

In [18]:
df_target_clean.head()

,CompanyNumber,CompanyName,CompanyStatus,CompanyCategory,CountryOfOrigin,RegAddress_Country,RegAddress_PostTown,RegAddress_PostCode,IncorporationDate,primary_sic_code,...,primary_sector,sector_id,Accounts_AccountCategory,Accounts_LastMadeUpDate,Accounts_NextDueDate,Mortgages_NumMortOutstanding,Mortgages_NumMortCharges,Mortgages_NumMortSatisfied,has_outstanding_charges,company_age_years
2789095,00000086,KENTSTONE PROPERTIES LIMITED,Active,Private Limited Company,United Kingdom,NaN,ASHFORD,TN25 6SX,1862-11-27,68209,...,Real Estate,4.0,SMALL,2025-03-31,2026-12-31,19,75,54,True,163.6
462163,00000118,ASHFORD CATTLE MARKET COMPANY LIMITED(THE),Active,Private Limited Company,United Kingdom,NaN,KENT,TN23 1DA,1856-09-25,46110,...,Wholesale & Retail,3.0,FULL,2025-07-31,2027-04-30,0,2,2,False,169.7
3758601,00000121,"ORIENTAL GAS COMPANY, LIMITED(THE)",Active,Private Limited Company,United Kingdom,NaN,LONDON,EC4A 2EA,1856-09-26,70100,...,"Technology, legal & professional",5.0,TOTAL EXEMPTION FULL,2025-06-30,2027-03-31,2,4,2,True,169.7
3501367,00000140,N & C BUILDING PRODUCTS LIMITED,Active,Private Limited Company,United Kingdom,NaN,ROMFORD,RM8 1SP,1856-09-30,20301,...,Manufacturing,2.0,FULL,2024-12-31,2026-09-30,1,5,4,True,169.7
3318822,00000295,METHODIST NEWSPAPER COMPANY LIMITED,Active,Private Limited Company,United Kingdom,ENGLAND,HUNSTANTON,PE36 6JG,1863-03-13,58130,...,"Technology, legal & professional",5.0,TOTAL EXEMPTION FULL,2026-03-22,2027-12-22,1,2,1,True,163.3


In [19]:
df_target_clean["CountryOfOrigin"].value_counts(dropna=False)

CountryOfOrigin
United Kingdom    3415686
UNITED KINGDOM          2
LUXEMBOURG              1
Name: count, dtype: int64

In [20]:
df_target_clean[ df_target_clean["CountryOfOrigin"]=='LUXEMBOURG']

,CompanyNumber,CompanyName,CompanyStatus,CompanyCategory,CountryOfOrigin,RegAddress_Country,RegAddress_PostTown,RegAddress_PostCode,IncorporationDate,primary_sic_code,...,primary_sector,sector_id,Accounts_AccountCategory,Accounts_LastMadeUpDate,Accounts_NextDueDate,Mortgages_NumMortOutstanding,Mortgages_NumMortCharges,Mortgages_NumMortSatisfied,has_outstanding_charges,company_age_years
5298384,SE000009,UBM INTERNATIONAL HOLDINGS UK SOCIETAS,Active,United Kingdom Societas,LUXEMBOURG,NaN,LONDON,SW1P 1WG,2008-10-08,70100,...,"Technology, legal & professional",5.0,AUDIT EXEMPTION SUBSIDIARY,2024-12-31,NaT,0,0,0,False,17.7


In [21]:
df_target_clean["CountryOfOrigin_clean"] = (
    df_target_clean["CountryOfOrigin"]
    .astype("string")
    .str.strip()
    .str.lower()
)

In [22]:
# filter UK
UK_COUNTRIES = {
    "england",
    "scotland",
    "wales",
    "northern ireland",
    "united kingdom",
}



df_target_clean["is_uk_company"] = df_target_clean["CountryOfOrigin_clean"].isin(UK_COUNTRIES)

df_target_clean["is_uk_company"].value_counts()


is_uk_company
True     3415688
False          1
Name: count, dtype: int64

In [23]:
len(df_target_clean)

3415689

In [24]:
df_target_clean

,CompanyNumber,CompanyName,CompanyStatus,CompanyCategory,CountryOfOrigin,RegAddress_Country,RegAddress_PostTown,RegAddress_PostCode,IncorporationDate,primary_sic_code,...,Accounts_AccountCategory,Accounts_LastMadeUpDate,Accounts_NextDueDate,Mortgages_NumMortOutstanding,Mortgages_NumMortCharges,Mortgages_NumMortSatisfied,has_outstanding_charges,company_age_years,CountryOfOrigin_clean,is_uk_company
2789095,00000086,KENTSTONE PROPERTIES LIMITED,Active,Private Limited Company,United Kingdom,NaN,ASHFORD,TN25 6SX,1862-11-27,68209,...,SMALL,2025-03-31,2026-12-31,19,75,54,True,163.6,united kingdom,True
462163,00000118,ASHFORD CATTLE MARKET COMPANY LIMITED(THE),Active,Private Limited Company,United Kingdom,NaN,KENT,TN23 1DA,1856-09-25,46110,...,FULL,2025-07-31,2027-04-30,0,2,2,False,169.7,united kingdom,True
3758601,00000121,"ORIENTAL GAS COMPANY, LIMITED(THE)",Active,Private Limited Company,United Kingdom,NaN,LONDON,EC4A 2EA,1856-09-26,70100,...,TOTAL EXEMPTION FULL,2025-06-30,2027-03-31,2,4,2,True,169.7,united kingdom,True
3501367,00000140,N & C BUILDING PRODUCTS LIMITED,Active,Private Limited Company,United Kingdom,NaN,ROMFORD,RM8 1SP,1856-09-30,20301,...,FULL,2024-12-31,2026-09-30,1,5,4,True,169.7,united kingdom,True
3318822,00000295,METHODIST NEWSPAPER COMPANY LIMITED,Active,Private Limited Company,United Kingdom,ENGLAND,HUNSTANTON,PE36 6JG,1863-03-13,58130,...,TOTAL EXEMPTION FULL,2026-03-22,2027-12-22,1,2,1,True,163.3,united kingdom,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3808063,ZC000169,PAIGNTON PIER COMPANY,Active,Other company type,United Kingdom,NaN,BARNSTAPLE,EX31 1SQ,1981-01-01,74990,...,DORMANT,2014-02-28,NaT,0,0,0,False,45.5,united kingdom,True
4876898,ZC000187,SUTTON HARBOUR COMPANY,Active,Other company type,United Kingdom,NaN,PLYMOUTH,PL4 0BN,1981-01-01,03110,...,FULL,2025-03-31,NaT,1,3,1,True,45.5,united kingdom,True
5540314,ZC000190,WHITSTABLE OYSTER FISHERY COMPANY (THE),Active,Other company type,United Kingdom,NaN,WHITSTABLE,CT5 1AB,1981-01-01,68209,...,TOTAL EXEMPTION FULL,2025-05-31,NaT,2,2,0,True,45.5,united kingdom,True
5012695,ZC000202,THE BRITISH STANDARDS INSTITUTION,Active,Other company type,United Kingdom,NaN,LONDON,WC2E 9RA,1929-04-22,70100,...,GROUP,2024-12-31,NaT,0,0,0,False,97.2,united kingdom,True


In [25]:
# Save
df_target_clean.to_csv(
    "../output/UKcompanies_8_sectors_cleaned.csv",
    index=False,
    encoding="utf-8-sig"
)

len(df_target_clean)


3415689